In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/Datasets/potato

testing  training  validation


In [ ]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# DATASET PATH
# ---------------------------------------------------------
TRAIN_PATH = r"/content/drive/MyDrive/Datasets/potato/training"
VAL_PATH = r"/content/drive/MyDrive/Datasets/potato/validation"
TEST_PATH = r"/content/drive/MyDrive/Datasets/potato/testing"

MODEL_SAVE_PATH = r"/content/drive/MyDrive/Datasets/potato/model_potato_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 4


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    # Check if the directory exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset directory not found: {path}")

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model: EfficientNetB0 + ConvNeXtTiny
# ---------------------------------------------------------
def build_hybrid_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("Loading datasets...")

    train_df = load_dataset(TRAIN_PATH)
    val_df = load_dataset(VAL_PATH)
    test_df = load_dataset(TEST_PATH)

    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        horizontal_flip=True
    )

    test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = test_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    test_images = test_gen.flow_from_dataframe(
        test_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False
    )

    print("Classes:", train_images.class_indices)

    model = build_hybrid_model(num_classes=len(train_images.class_indices))

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
    ]

    print("Training...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    print("Evaluating...")
    loss, acc = model.evaluate(test_images, verbose=0)
    print(f"Test Accuracy: {acc * 100:.2f}%")

    # Accuracy plot
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/potato/accuracy.png")
    plt.close()

    # Loss plot
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/potato/loss.png")
    plt.close()

train()

Loading datasets...
Found 3252 validated image filenames belonging to 3 classes.
Found 416 validated image filenames belonging to 3 classes.
Found 405 validated image filenames belonging to 3 classes.
Classes: {'Early_Blight': 0, 'Healthy': 1, 'Late_Blight': 2}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/4
102/102 ━━━━━━━━━━━━━━━━━━━━ 1491s 14s/step - accuracy: 0.6951 - loss: 0.7013 - val_accuracy: 0.9303 - val_loss: 0.2070
Epoch 2/4
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 198ms/step - accuracy: 0.9277 - loss: 0.1989 - val_accuracy: 0.9639 - val_loss: 0.1231
Epoch 3/4
102/102 ━━━━━━━━━━━━━━━━━━━━ 22s 214ms/step - accuracy: 0.9603 - loss: 0.1298 - val_accuracy: 0.9712 - val_loss: 0.1057
Epoch 4/4
102/102 ━━━━━━━━━━━━━━━━━━━━ 21s 210ms/step - accuracy: 0.9628 - loss: 0.1196 - val_accuracy: 0.9856 - val_loss: 0.0721
Saving model...
Evaluating...
Test Accuracy: 97.78%
